# AADS-ULoRA Colab: Data Preparation

This notebook handles data download, preprocessing, and dataset creation for the AADS-ULoRA training pipeline.

## What this notebook does:
1. **Mount Google Drive** - For persistent storage
2. **Download datasets** - PlantVillage and additional data
3. **Preprocess images** - Resize, normalize, augment
4. **Create splits** - Train/validation/test splits
5. **Validate data** - Check for corrupt files and class balance

---
**Expected Runtime:** 10-30 minutes (depending on dataset size)
**Required:** Google Drive with at least 20GB free space

---

## 1. Mount Google Drive

Mount your Google Drive to persist data across Colab sessions.

In [ ]:
from google.colab import drive
import os
from pathlib import Path

def gate_check(step_id: str, check_name: str, condition: bool, expected: str, actual: str):
    status = "PASS" if condition else "FAIL"
    print(f"[{step_id}] {status} :: {check_name} | expected={expected} | actual={actual}")
    if not condition:
        raise RuntimeError(f"Gate failed at {step_id}::{check_name}")

print("🔗 Mounting Google Drive...")
drive.mount('/content/drive')

# Create workspace directory
workspace_dir = Path('/content/aads_ulora')
workspace_dir.mkdir(parents=True, exist_ok=True)

# Set environment variable for workspace
os.environ['AADS_WORKSPACE'] = str(workspace_dir)

gate_check('DATA_SETUP', 'workspace_exists', workspace_dir.exists(), 'workspace directory exists', str(workspace_dir))
print(f"✅ Workspace directory: {workspace_dir}")

## 2. Install Dependencies

Install all required packages for data processing.

In [ ]:
import sys
import subprocess

def install_package(package):
    print(f"Installing {package}...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

# Install required packages
packages = [
    'torch', 'torchvision', 'torchaudio',
    'transformers', 'peft', 'accelerate',
    'numpy', 'pandas', 'pillow',
    'scikit-learn', 'tqdm', 'psutil',
    'gdown'
]

for pkg in packages:
    try:
        __import__(pkg.split('>')[0].split('[')[0])
        print(f"✅ {pkg} already installed")
    except ImportError:
        install_package(pkg)

print("\n✅ All dependencies installed!")

## 3. Download Data

Download PlantVillage dataset and any additional data needed for training.

In [ ]:
from pathlib import Path
import logging

from scripts.download_data_colab import DownloadConfig, DriveDownloader
from src.core.colab_contract import is_placeholder_drive_id

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Data directory
data_dir = workspace_dir / 'data'
data_dir.mkdir(exist_ok=True)

# Google Drive file IDs for datasets (replace with actual IDs)
datasets = {
    'plantvillage': {
        'file_id': 'YOUR_FILE_ID_HERE',
        'filename': 'plantvillage.zip',
    }
}

# Fail-fast ID validation
gate_check(
    'DATA_IDS_VALID',
    'all_dataset_ids_set',
    all(not is_placeholder_drive_id(info['file_id']) for info in datasets.values()),
    'all dataset file_id values are real Google Drive IDs',
    str({k: v['file_id'] for k, v in datasets.items()})
)

downloader = DriveDownloader(DownloadConfig(download_dir=str(data_dir), max_retries=3))

# Download all datasets
downloaded_files = {}
for name, info in datasets.items():
    downloaded_files[name] = downloader.download_file(
        file_id=info['file_id'],
        destination=info['filename'],
        description=name,
    )

gate_check(
    'DATA_SCHEMA_OK',
    'downloaded_files_present',
    all(path is not None and Path(path).exists() for path in downloaded_files.values()),
    'all datasets downloaded successfully',
    str(downloaded_files)
)

print("\n✅ Data download complete!")

## 4. Extract and Organize Data

Extract zip files and organize data into proper directory structure.

In [ ]:
import zipfile
import shutil
from tqdm import tqdm

def extract_zip(zip_path, extract_to):
    """Extract a zip file with progress bar."""
    extract_to = Path(extract_to)
    extract_to.mkdir(parents=True, exist_ok=True)

    print(f"Extracting {zip_path} to {extract_to}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for member in tqdm(zip_ref.infolist(), desc='Extracting'):
            zip_ref.extract(member, extract_to)

    print(f"✅ Extraction complete")

# Extract datasets
for name, filepath in downloaded_files.items():
    if filepath and Path(filepath).suffix == '.zip':
        extract_dir = data_dir / name
        extract_zip(filepath, extract_dir)

gate_check(
    'DATA_SCHEMA_OK',
    'extracted_dirs_present',
    all((data_dir / name).exists() for name in downloaded_files.keys()),
    'each downloaded dataset extracted',
    str([str(data_dir / name) for name in downloaded_files.keys()])
)

print("\n✅ All data extracted!")

## 5. Preprocess Data

Resize images, normalize, and create train/val/test splits.

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets
from PIL import Image
import numpy as np

# Image transformations
image_size = 224

train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Assuming data is organized by class folders
train_data_path = data_dir / 'plantvillage' / 'train'
val_data_path = data_dir / 'plantvillage' / 'val'
test_data_path = data_dir / 'plantvillage' / 'test'

print("✅ Transformations defined")
print(f"Train data path: {train_data_path}")
print(f"Val data path: {val_data_path}")
print(f"Test data path: {test_data_path}")

## 6. Validate Dataset

Check dataset integrity and class distribution.

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

def validate_dataset(data_path, transform):
    """Validate dataset and print statistics."""
    if not data_path.exists():
        print(f"⚠️  Path does not exist: {data_path}")
        return None
    
    dataset = datasets.ImageFolder(root=data_path, transform=transform)
    
    print(f"\nDataset: {data_path.name}")
    print(f"  Total samples: {len(dataset)}")
    print(f"  Number of classes: {len(dataset.classes)}")
    print(f"  Classes: {dataset.classes}")
    
    # Count samples per class
    class_counts = Counter([label for _, label in dataset])
    print("  Class distribution:")
    for class_idx, count in sorted(class_counts.items()):
        class_name = dataset.classes[class_idx]
        print(f"    {class_name}: {count}")
    
    return dataset

# Validate each split
train_dataset = validate_dataset(train_data_path, train_transform)
val_dataset = validate_dataset(val_data_path, val_transform)
test_dataset = validate_dataset(test_data_path, val_transform)

print("\n✅ Dataset validation complete!")

## 7. Visualize Samples

Display sample images from the dataset to verify preprocessing.

In [ ]:
def visualize_samples(dataset, num_samples=5):
    """Display sample images from dataset."""
    if dataset is None:
        print("Dataset not available")
        return
    
    fig, axes = plt.subplots(1, num_samples, figsize=(15, 3))
    
    indices = np.random.choice(len(dataset), num_samples, replace=False)
    
    for idx, ax in zip(indices, axes):
        image, label = dataset[idx]
        
        # Denormalize for visualization
        image = image.numpy().transpose(1, 2, 0)
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        image = std * image + mean
        image = np.clip(image, 0, 1)
        
        ax.imshow(image)
        ax.set_title(dataset.classes[label])
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize training samples
print("Training samples:")
visualize_samples(train_dataset)

print("\nValidation samples:")
visualize_samples(val_dataset)

## 8. Save Dataset Metadata

Save dataset information for later use in training.

In [ ]:
import json

# Collect dataset metadata
metadata = {
    'train': {
        'num_samples': len(train_dataset) if train_dataset else 0,
        'num_classes': len(train_dataset.classes) if train_dataset else 0,
        'classes': train_dataset.classes if train_dataset else []
    } if train_dataset else None,
    'val': {
        'num_samples': len(val_dataset) if val_dataset else 0,
        'num_classes': len(val_dataset.classes) if val_dataset else 0,
        'classes': val_dataset.classes if val_dataset else []
    } if val_dataset else None,
    'test': {
        'num_samples': len(test_dataset) if test_dataset else 0,
        'num_classes': len(test_dataset.classes) if test_dataset else 0,
        'classes': test_dataset.classes if test_dataset else []
    } if test_dataset else None,
    'image_size': image_size,
    'normalize_mean': [0.485, 0.456, 0.406],
    'normalize_std': [0.229, 0.224, 0.225]
}

# Save metadata
metadata_path = workspace_dir / 'data' / 'dataset_metadata.json'
metadata_path.parent.mkdir(parents=True, exist_ok=True)

with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Dataset metadata saved to: {metadata_path}")
print("\nData preparation complete! Ready for training.")